In [2]:
import pandas as pd
from maricovault.MaricoDB import MaricoSnowflake

def get_dbconnection(db_name):    

    KEY_VAULT_NAME = "prod-pwd"

    if db_name == 'PROD':
        db_name = 'prod'
    else:
        db_name = 'dev'
    

    msf = MaricoSnowflake(KEY_VAULT_NAME)
    msf.get_db_credentials(db_name=db_name)
    msf.connect()
    dbconnection = msf.get_connection()
    
    return dbconnection


def read_qtr_ind_rate_table():
    """
    Fetch the club sku information from  DWH_SAP_INDEX_TURNOVER_MONTHWISE table.

    Return:
        qtr_ind_rate_data: pandas dataframe
        - dataframe contains all_df the results from the index rate table.
    """
    connection = get_dbconnection(db_name='PROD')
    query = """select * from DWH_SAP_INDEX_TURNOVER_MONTHWISE 
                where latest_rate_flag=1 and company_code='MIL'"""
    qtr_ind_rate_data = pd.read_sql(con=connection, sql=query)
    qtr_ind_rate_data.columns = qtr_ind_rate_data.columns.str.lower()
    qtr_ind_rate =  qtr_ind_rate_data[['date', 'brand_code', 'turnover']]
    qtr_ind_rate = qtr_ind_rate.rename(columns= {'date':'month_date', 'turnover':'qtr_ind_rate'})
    connection.close()
    return qtr_ind_rate


dev_conn = get_dbconnection('DEV')
prod_conn = get_dbconnection('PROD')


Credentials retrieved successfully for dev db.

Credentials retrieved successfully for prod db.


In [3]:
model = pd.read_csv("/data/aman_singh/acuuracy_check/mt_channels Live Run aug'26.csv")
missing = pd.read_csv("/data/aman_singh/acuuracy_check/missing_df_mt_all.csv")

In [4]:
print(model['M month'].unique())

['0' 'M' 'M+1' 'M+2' 'M+3' 'M+4' 'M+5' 'M+6' 'M+7']


In [5]:
print(missing['M month'].unique())

['0' 'M' 'M+1' 'M+2' 'M+3' 'M+4' 'M+5' 'M+6' 'M+7']


In [6]:
model['key'].nunique()

4254

In [7]:
#missing = missing[missing['M month'].notna()]

In [8]:
print(missing['M month'].unique())

['0' 'M' 'M+1' 'M+2' 'M+3' 'M+4' 'M+5' 'M+6' 'M+7']


In [9]:
model['skipped'] = 0
missing['skipped'] = 1

In [10]:
all_df = pd.concat([
    model, missing
], ignore_index=True)

In [11]:
all_df['channel'].unique()

array(['MT'], dtype=object)

In [12]:
all_df.duplicated(subset=['channel', 'key', 'month_date']).sum()

0

In [13]:
import pandas as pd
import numpy as np
import pymannkendall as mk

def detect_trend_for_group(df_grp):
    """
    Detect final trend flag and p3m_slope_flag separately.
    Must contain 'month_date', 'vol_in_rum', 'run_month'
    """

    # ---------- 1. Sort ----------
    df_grp = df_grp.sort_values("month_date")

    # ---------- 2. Identify run_month ----------
    run_month = df_grp["run_month"].max()

    # actual data = months < run_month
    df_actual = df_grp[df_grp["month_date"] < run_month]

    # if no actual data → no trend
    if df_actual.empty or len(df_actual) < 4:
        return pd.Series({"trend_flag": 0, "p3m_slope_flag": 0})

    # ---------- 3. MK Trend ----------
    series = df_actual["sec_vol_actuals_rum_month_value"].astype(float)

    try:
        mk_result = mk.original_test(series)
        if mk_result.trend == "increasing":
            mk_trend = 1
        elif mk_result.trend == "decreasing":
            mk_trend = -1
        else:
            mk_trend = 0
    except:
        mk_trend = 0

    # ---------- 4. P3M Slope ----------
    # last 4 months → take last 3 with shift
    #shifted_series = series.shift(1).dropna()

    p3m_values = series.tail(3).values
    #print(p3m_values)

    if len(p3m_values) < 3:
        slope_flag = 0
    else:
        x = np.arange(3)
        slope = np.polyfit(x, p3m_values, 1)[0]
        slope_flag = 1 if slope > 0 else (-1 if slope < 0 else 0)
        

    return pd.Series({
        "trend_flag": mk_trend,
        "p3m_slope_flag": slope_flag
    })


# ---------------------------------------------------------
# APPLY ON ENTIRE DATASET
# ---------------------------------------------------------

# trend_df = final_df.groupby(
#     ["platform_name", "parent_material_code"]
# ).apply(detect_trend_for_group).reset_index()

# trend_df = final_df[final_df['key'] == 'Zepto_721898'].groupby(
#     ["platform_name", "parent_material_code", "run_month"]
# ).apply(detect_trend_for_group).reset_index()
trend_df = all_df.groupby(
    ['channel','key',"run_month"]
).apply(detect_trend_for_group).reset_index()

In [14]:
trend_df["final_trend"] = np.where(
    (trend_df["trend_flag"] == 1) & (trend_df["p3m_slope_flag"] == 1), 1,
    np.where(
        (trend_df["trend_flag"] == -1) & (trend_df["p3m_slope_flag"] == -1), -1,
        0
    )
)
trend_df

,channel,key,run_month,trend_flag,p3m_slope_flag,final_trend
0,MT,BCE1_D231_718303,2026-08-31,-1,1,0
1,MT,BCE1_D231_718312,2026-08-31,0,-1,0
2,MT,BCE1_D231_718317,2026-08-31,0,-1,0
3,MT,BCE1_D231_718318,2026-08-31,-1,-1,-1
4,MT,BCE1_D231_718322,2026-08-31,-1,0,0
...,...,...,...,...,...,...
7376,MT,MCW2_D463_811097,2026-08-31,0,1,0
7377,MT,MCW2_D463_811100,2026-08-31,0,1,0
7378,MT,MCW2_D463_811218,2026-08-31,0,1,0
7379,MT,MCW2_D463_811219,2026-08-31,0,1,0


## detect seasonality

In [15]:
from statsmodels.tsa.stattools import acf
import numpy as np
import pandas as pd

def detect_yearly_seasonality(df_grp, threshold=0.3):
    """
    Detects yearly seasonality using ACF at lag=12 only.
    Uses vol_in_rum as the metric.
    """
    df_grp = df_grp.sort_values("month_date")
    run_month = df_grp["run_month"].max()

    # actual data = months < run_month
    df_actual = df_grp[df_grp["month_date"] < run_month]
    series = df_actual["sec_vol_actuals_rum_month_value"].astype(float).values

    # Need at least 18 points to compare last year vs this year
    if len(series) < 18:
        return 0

    # Compute ACF up to lag-12
    acf_vals = acf(series, nlags=12, fft=False)

    lag12_acf = acf_vals[12]

    # absolute ACF because seasonal correlation can be negative as well
    if abs(lag12_acf) >= threshold:
        return 1
    else:
        return 0
    

seasonality_df = all_df.groupby(
    ['channel','ASM', 'Depot', 'brand_code',"run_month"]
).apply(detect_yearly_seasonality).reset_index(name="seasonality_flag")

seasonality_df



/data/aman_singh/.env_myvenv/lib/python3.10/site-packages/statsmodels/tsa/stattools.py:702: RuntimeWarning: invalid value encountered in divide
  acf = avf[: nlags + 1] / avf[0]
/data/aman_singh/.env_myvenv/lib/python3.10/site-packages/statsmodels/tsa/stattools.py:702: RuntimeWarning: invalid value encountered in divide
  acf = avf[: nlags + 1] / avf[0]
/data/aman_singh/.env_myvenv/lib/python3.10/site-packages/statsmodels/tsa/stattools.py:702: RuntimeWarning: invalid value encountered in divide
  acf = avf[: nlags + 1] / avf[0]
/data/aman_singh/.env_myvenv/lib/python3.10/site-packages/statsmodels/tsa/stattools.py:702: RuntimeWarning: invalid value encountered in divide
  acf = avf[: nlags + 1] / avf[0]


,channel,ASM,Depot,brand_code,run_month,seasonality_flag
0,MT,BCE1,D231,H&C,2026-08-31,0
1,MT,BCE1,D231,H&C_ALMND,2026-08-31,0
2,MT,BCE1,D231,LIVON S-R,2026-08-31,0
3,MT,BCE1,D231,NHR-SABDM,2026-08-31,0
4,MT,BCE1,D231,NHR_VTEHO,2026-08-31,0
...,...,...,...,...,...,...
2045,MT,MCW2,D463,SF_MNCHPS,2026-08-31,0
2046,MT,MCW2,D463,SF_SOYACN,2026-08-31,0
2047,MT,MCW2,D463,SW HRGEL,2026-08-31,0
2048,MT,MCW2,D463,SW_HR_WAX,2026-08-31,0


In [16]:
import numpy as np
import pandas as pd

def compute_thresholds(df_grp):
    """
    df_grp MUST contain:
    - month_date
    - vol_in_rum
    - run_month

    Returns: lower_threshold, upper_threshold, mean, std
    """

    df_grp = df_grp.sort_values("month_date")
    run_month = df_grp["run_month"].max()

    # --- Use ONLY actual data (strictly before run month)
    df_actual = df_grp[df_grp["month_date"] < run_month]

    series = df_actual["sec_vol_actuals_rum_month_value"].astype(float).values

    # If no real data → return zeros
    if len(series) == 0:
        return pd.Series({
            "lower_threshold": 0,
            "upper_threshold": 0,
            "mean_value": 0,
            "std_value": 0
        })

    # --- Take last 12 months OR all_df available
    if len(series) > 12:
        series = series[-12:]

    mean_val = np.mean(series)
    std_val = np.std(series)

    # --- SPECIAL CASE: ≤3 data points
    if len(series) <= 3:
        lower = 0.5 * mean_val
        upper = 2 * mean_val

        return pd.Series({
            "lower_threshold": lower,
            "upper_threshold": upper,
            "mean_value": mean_val,
            "std_value": std_val
        })

    # --- Normal case (std can be zero also)
    lower = max(0,mean_val - 2*std_val)
    upper = mean_val + 3*std_val

    return pd.Series({
        "lower_threshold": lower,
        "upper_threshold": upper,
        "mean_value": mean_val,
        "std_value": std_val
    })

threshold_df = all_df.groupby(
    ["channel", "key","run_month"]
).apply(compute_thresholds).reset_index()

threshold_df.head()


,channel,key,run_month,lower_threshold,upper_threshold,mean_value,std_value
0,MT,BCE1_D231_718303,2026-08-31,0.0,0.001561,0.000490,0.000357
1,MT,BCE1_D231_718312,2026-08-31,0.0,0.005136,0.001921,0.001072
2,MT,BCE1_D231_718317,2026-08-31,0.0,0.002326,0.000745,0.000527
3,MT,BCE1_D231_718318,2026-08-31,0.0,0.002981,0.000699,0.000761
4,MT,BCE1_D231_718322,2026-08-31,0.0,0.003313,0.000394,0.000973


In [17]:
trend_df = trend_df.merge(threshold_df, on = ["channel", "key","run_month"], how = 'left')
trend_df

,channel,key,run_month,trend_flag,p3m_slope_flag,final_trend,lower_threshold,upper_threshold,mean_value,std_value
0,MT,BCE1_D231_718303,2026-08-31,-1,1,0,0.0,0.001561,0.000490,0.000357
1,MT,BCE1_D231_718312,2026-08-31,0,-1,0,0.0,0.005136,0.001921,0.001072
2,MT,BCE1_D231_718317,2026-08-31,0,-1,0,0.0,0.002326,0.000745,0.000527
3,MT,BCE1_D231_718318,2026-08-31,-1,-1,-1,0.0,0.002981,0.000699,0.000761
4,MT,BCE1_D231_718322,2026-08-31,-1,0,0,0.0,0.003313,0.000394,0.000973
...,...,...,...,...,...,...,...,...,...,...
7376,MT,MCW2_D463_811097,2026-08-31,0,1,0,0.0,0.001694,0.000418,0.000425
7377,MT,MCW2_D463_811100,2026-08-31,0,1,0,0.0,0.001280,0.000395,0.000295
7378,MT,MCW2_D463_811218,2026-08-31,0,1,0,0.0,0.002969,0.001153,0.000605
7379,MT,MCW2_D463_811219,2026-08-31,0,1,0,0.0,0.004295,0.001183,0.001038


In [18]:
# trend_df.to_csv('t_thres_df_2.csv')

In [19]:
all_df.shape

(253316, 109)

In [20]:
all_df = all_df.merge(seasonality_df, on = ['channel','ASM', 'Depot', 'brand_code',"run_month"], how = 'left')
all_df

,key,month_date,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,...,print_spends_in_lacs,tv_spends_in_lacs,radio_spends_in_lacs,npt,pt,npd,update_timestamp,drive,outlier,seasonality_flag
0,BCS1_D530_718472,2023-01-31,57.6,99.9,137.941975,21.24,0.002862,0.004963,0.006853,0.001055,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
1,BCS1_D530_718472,2023-02-28,57.6,99.9,134.940300,21.84,0.002862,0.004963,0.006704,0.001085,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
2,BCS1_D530_718472,2023-03-31,57.6,99.9,163.988721,21.60,0.002862,0.004963,0.008147,0.001073,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
3,BCS1_D530_718472,2023-04-30,57.6,99.9,166.389106,22.02,0.002862,0.004963,0.008267,0.001094,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
4,BCS1_D530_718472,2023-05-31,41.4,99.9,126.807418,21.78,0.002057,0.004963,0.006300,0.001082,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
253311,MCW2_D463_811220,2026-11-30,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
253312,MCW2_D463_811220,2026-12-31,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
253313,MCW2_D463_811220,2027-01-31,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
253314,MCW2_D463_811220,2027-02-28,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0


In [21]:
trend_df.columns

Index(['channel', 'key', 'run_month', 'trend_flag', 'p3m_slope_flag',
       'final_trend', 'lower_threshold', 'upper_threshold', 'mean_value',
       'std_value'],
      dtype='object')

In [22]:
all_df = all_df.merge(trend_df[["channel", "key","run_month",
                                    'final_trend','lower_threshold', 'upper_threshold']], on = ["channel", "key","run_month"], how = 'left')
all_df


,key,month_date,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,...,npt,pt,npd,update_timestamp,drive,outlier,seasonality_flag,final_trend,lower_threshold,upper_threshold
0,BCS1_D530_718472,2023-01-31,57.6,99.9,137.941975,21.24,0.002862,0.004963,0.006853,0.001055,...,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0.0,0.009557
1,BCS1_D530_718472,2023-02-28,57.6,99.9,134.940300,21.84,0.002862,0.004963,0.006704,0.001085,...,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0.0,0.009557
2,BCS1_D530_718472,2023-03-31,57.6,99.9,163.988721,21.60,0.002862,0.004963,0.008147,0.001073,...,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0.0,0.009557
3,BCS1_D530_718472,2023-04-30,57.6,99.9,166.389106,22.02,0.002862,0.004963,0.008267,0.001094,...,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0.0,0.009557
4,BCS1_D530_718472,2023-05-31,41.4,99.9,126.807418,21.78,0.002057,0.004963,0.006300,0.001082,...,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0.0,0.009557
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
253311,MCW2_D463_811220,2026-11-30,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0.0,0.002728
253312,MCW2_D463_811220,2026-12-31,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0.0,0.002728
253313,MCW2_D463_811220,2027-01-31,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0.0,0.002728
253314,MCW2_D463_811220,2027-02-28,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0.0,0.002728


In [23]:
all_df = all_df.sort_values(['channel','key', 'month_date'])

# base LY


# LY lags
all_df['ly_lag1_value'] = (
    all_df
    .groupby(['channel','key'])['sec_vol_actuals_rum_month_value']
    .shift(13)
)

all_df['ly_lag2_value'] = (
    all_df
    .groupby(['channel','key'])['sec_vol_actuals_rum_month_value']
    .shift(14)
)

# LY leads
all_df['ly_lead1_value'] = (
    all_df
    .groupby(['channel','key'])['sec_vol_actuals_rum_month_value']
    .shift(11)
)

all_df['ly_lead2_value'] = (
    all_df
    .groupby(['channel','key'])['sec_vol_actuals_rum_month_value']
    .shift(10)
)


In [24]:
all_df[all_df.select_dtypes(include='number').columns] = all_df.select_dtypes(include='number').fillna(0)
all_df

,key,month_date,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,...,drive,outlier,seasonality_flag,final_trend,lower_threshold,upper_threshold,ly_lag1_value,ly_lag2_value,ly_lead1_value,ly_lead2_value
93884,BCE1_D231_718303,2023-01-31,0.018667,0.037167,0.041426,0.0112,0.000463,0.000921,0.001027,0.000278,...,0.0,0.0,0,0,0.0,0.001561,0.000000,0.0,0.000000,0.000000
93885,BCE1_D231_718303,2023-02-28,0.018667,0.037167,0.039172,0.0084,0.000463,0.000921,0.000971,0.000208,...,0.0,0.0,0,0,0.0,0.001561,0.000000,0.0,0.000000,0.000000
93886,BCE1_D231_718303,2023-03-31,0.018667,0.037167,0.039450,0.0140,0.000463,0.000921,0.000978,0.000347,...,0.0,0.0,0,0,0.0,0.001561,0.000000,0.0,0.000000,0.000000
93887,BCE1_D231_718303,2023-04-30,0.018667,0.037167,0.042242,0.0182,0.000463,0.000921,0.001047,0.000451,...,0.0,0.0,0,0,0.0,0.001561,0.000000,0.0,0.000000,0.000000
93888,BCE1_D231_718303,2023-05-31,0.037333,0.037167,0.046562,0.0280,0.000926,0.000921,0.001154,0.000694,...,0.0,0.0,0,0,0.0,0.001561,0.000000,0.0,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
253311,MCW2_D463_811220,2026-11-30,0.000000,0.000000,0.000000,0.0000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0,0,0.0,0.002728,0.000000,0.0,0.000000,0.000000
253312,MCW2_D463_811220,2026-12-31,0.000000,0.000000,0.000000,0.0000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0,0,0.0,0.002728,0.000000,0.0,0.000000,0.001624
253313,MCW2_D463_811220,2027-01-31,0.000000,0.000000,0.000000,0.0000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0,0,0.0,0.002728,0.000000,0.0,0.001624,0.000000
253314,MCW2_D463_811220,2027-02-28,0.000000,0.000000,0.000000,0.0000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0,0,0.0,0.002728,0.000000,0.0,0.000000,0.000812


In [25]:
# all_df.to_csv('break.csv')

In [26]:
all_df

,key,month_date,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,...,drive,outlier,seasonality_flag,final_trend,lower_threshold,upper_threshold,ly_lag1_value,ly_lag2_value,ly_lead1_value,ly_lead2_value
93884,BCE1_D231_718303,2023-01-31,0.018667,0.037167,0.041426,0.0112,0.000463,0.000921,0.001027,0.000278,...,0.0,0.0,0,0,0.0,0.001561,0.000000,0.0,0.000000,0.000000
93885,BCE1_D231_718303,2023-02-28,0.018667,0.037167,0.039172,0.0084,0.000463,0.000921,0.000971,0.000208,...,0.0,0.0,0,0,0.0,0.001561,0.000000,0.0,0.000000,0.000000
93886,BCE1_D231_718303,2023-03-31,0.018667,0.037167,0.039450,0.0140,0.000463,0.000921,0.000978,0.000347,...,0.0,0.0,0,0,0.0,0.001561,0.000000,0.0,0.000000,0.000000
93887,BCE1_D231_718303,2023-04-30,0.018667,0.037167,0.042242,0.0182,0.000463,0.000921,0.001047,0.000451,...,0.0,0.0,0,0,0.0,0.001561,0.000000,0.0,0.000000,0.000000
93888,BCE1_D231_718303,2023-05-31,0.037333,0.037167,0.046562,0.0280,0.000926,0.000921,0.001154,0.000694,...,0.0,0.0,0,0,0.0,0.001561,0.000000,0.0,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
253311,MCW2_D463_811220,2026-11-30,0.000000,0.000000,0.000000,0.0000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0,0,0.0,0.002728,0.000000,0.0,0.000000,0.000000
253312,MCW2_D463_811220,2026-12-31,0.000000,0.000000,0.000000,0.0000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0,0,0.0,0.002728,0.000000,0.0,0.000000,0.001624
253313,MCW2_D463_811220,2027-01-31,0.000000,0.000000,0.000000,0.0000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0,0,0.0,0.002728,0.000000,0.0,0.001624,0.000000
253314,MCW2_D463_811220,2027-02-28,0.000000,0.000000,0.000000,0.0000,0.000000,0.000000,0.000000,0.000000,...,0.0,0.0,0,0,0.0,0.002728,0.000000,0.0,0.000000,0.000812


In [27]:
brand_seas = pd.read_excel('/data/aman_singh/acuuracy_check/seasonality.xlsx', sheet_name = 'brand')
brand_seas.columns = brand_seas.columns.str.lower()
brand_seas.rename(columns={'brand':'brand_code', 'months_num':'month', 'flag':'is_seasonal_month'}, inplace=True)

all_df['month_date'] = pd.to_datetime(all_df['month_date'])
all_df['month'] = all_df['month_date'].dt.month
all_df = all_df.merge(brand_seas, on = ['brand_code', 'month'], how = 'left')
all_df['is_seasonal_month'].fillna(0, inplace=True)

psku_seas = pd.read_excel('/data/aman_singh/acuuracy_check/seasonality.xlsx', sheet_name = 'psku')
psku_seas.columns = psku_seas.columns.str.lower()
psku_seas.rename(columns={'months_num':'month', 'flag':'is_seasonal_month_psku'}, inplace=True)

all_df = all_df.merge(psku_seas[['parent_material_code', 'month','is_seasonal_month_psku']], on = ['parent_material_code', 'month'], how = 'left')
all_df['is_seasonal_month_psku'].fillna(0, inplace=True)
all_df['final_seasonal_month'] = np.where(
    (all_df['is_seasonal_month'] == 1) | (all_df['is_seasonal_month_psku'] == 1), 1, 0
)



/tmp/ipykernel_3195255/3019102920.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  all_df['month'] = all_df['month_date'].dt.month


In [28]:
all_df['run_month'] = pd.to_datetime(all_df['run_month'])
def compute_adjusted_pm(df_grp, window, column, year_shift=0):
    df_grp = df_grp.sort_values("month_date")

    run_month = df_grp["run_month"].iloc[0]

    # define cutoff
    end_date = run_month - pd.DateOffset(years=year_shift)

    # keep only eligible history (before run month & non-event)
    hist = df_grp[
        (df_grp["month_date"] < end_date) &
        (df_grp["final_seasonal_month"] == 0)
    ]

    if hist.empty:
        return np.nan

    # take last `window` non-event months
    hist = hist.tail(window)

    # if len(hist) < window:
    #     return np.nan   # optional, keeps behavior strict

    return hist[column].mean()

adj_df = all_df.groupby(
    ['key', "run_month"]
).apply(
    lambda x: pd.Series({
        "P3M_non_seasonal": compute_adjusted_pm(x, 3,'sec_vol_actuals_rum_month',0),
        "P6M_non_seasonal": compute_adjusted_pm(x, 6,'sec_vol_actuals_rum_month',0),
        "P3M_non_seasonal_value": compute_adjusted_pm(x, 3,'sec_vol_actuals_rum_month_value',0),
        "P6M_non_seasonal_value": compute_adjusted_pm(x, 6,'sec_vol_actuals_rum_month_value',0)
    })
).reset_index()
adj_df


adj_ly_df = all_df.groupby(
    ['key', "run_month"]
).apply(
    lambda x: pd.Series({
        "LY_P3M_non_seasonal": compute_adjusted_pm(x, 3,'sec_vol_actuals_rum_month',year_shift=1),
        "LY_P6M_non_seasonal": compute_adjusted_pm(x, 6,'sec_vol_actuals_rum_month', year_shift=1),
        "LY_P3M_non_seasonal_value": compute_adjusted_pm(x, 3,'sec_vol_actuals_rum_month_value', year_shift=1),
        "LY_P6M_non_seasonal_value": compute_adjusted_pm(x, 6,"sec_vol_actuals_rum_month_value", year_shift=1)
    })
).reset_index()

adj_df = adj_df.merge(adj_ly_df, on = ['key', 'run_month'], how = 'left')
#adj_df[adj_df['key'] == 'reliance_b2c_2_haryana_718488']
adj_df

,key,run_month,P3M_non_seasonal,P6M_non_seasonal,P3M_non_seasonal_value,P6M_non_seasonal_value,LY_P3M_non_seasonal,LY_P6M_non_seasonal,LY_P3M_non_seasonal_value,LY_P6M_non_seasonal_value
0,BCE1_D231_718303,2026-08-31,0.009333,0.020833,0.000231,0.000517,0.009333,0.016333,0.000231,0.000405
1,BCE1_D231_718312,2026-08-31,0.020000,0.043333,0.000699,0.001514,0.066667,0.033333,0.002328,0.001164
2,BCE1_D231_718317,2026-08-31,24.000000,16.800000,0.000931,0.000652,62.400000,31.200000,0.002422,0.001211
3,BCE1_D231_718318,2026-08-31,28.800000,24.000000,0.001118,0.000931,57.600000,28.800000,0.002235,0.001118
4,BCE1_D231_718322,2026-08-31,0.000000,0.000000,0.000000,0.000000,0.180000,0.123333,0.003039,0.002082
...,...,...,...,...,...,...,...,...,...,...
7376,MCW2_D463_811097,2026-08-31,0.018667,0.011833,0.000659,0.000418,NaN,NaN,NaN,NaN
7377,MCW2_D463_811100,2026-08-31,0.015333,0.011200,0.000541,0.000395,NaN,NaN,NaN,NaN
7378,MCW2_D463_811218,2026-08-31,0.031000,0.032667,0.001094,0.001153,NaN,NaN,NaN,NaN
7379,MCW2_D463_811219,2026-08-31,0.051667,0.033500,0.001824,0.001183,NaN,NaN,NaN,NaN


In [29]:
#adj_df.to_csv('seasonal_p3m_mt.csv', index=False)
#all_df[all_df['month_date'].isin(['2025-11-30','2025-12-31','2026-01-31'])].groupby(['key','run_month','month_date','final_seasonal_month'])['vol_in_rum'].sum().reset_index().to_csv('seasonal_month_check.csv', index=False)
all_df = all_df.merge(
    adj_df,
    on=['key','run_month'],
    how="left"
)
all_df

,key,month_date,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,...,is_seasonal_month_psku,final_seasonal_month,P3M_non_seasonal,P6M_non_seasonal,P3M_non_seasonal_value,P6M_non_seasonal_value,LY_P3M_non_seasonal,LY_P6M_non_seasonal,LY_P3M_non_seasonal_value,LY_P6M_non_seasonal_value
0,BCE1_D231_718303,2023-01-31,0.018667,0.037167,0.041426,0.0112,0.000463,0.000921,0.001027,0.000278,...,0.0,0,0.009333,0.020833,0.000231,0.000517,0.009333,0.016333,0.000231,0.000405
1,BCE1_D231_718303,2023-02-28,0.018667,0.037167,0.039172,0.0084,0.000463,0.000921,0.000971,0.000208,...,0.0,0,0.009333,0.020833,0.000231,0.000517,0.009333,0.016333,0.000231,0.000405
2,BCE1_D231_718303,2023-03-31,0.018667,0.037167,0.039450,0.0140,0.000463,0.000921,0.000978,0.000347,...,0.0,1,0.009333,0.020833,0.000231,0.000517,0.009333,0.016333,0.000231,0.000405
3,BCE1_D231_718303,2023-04-30,0.018667,0.037167,0.042242,0.0182,0.000463,0.000921,0.001047,0.000451,...,0.0,1,0.009333,0.020833,0.000231,0.000517,0.009333,0.016333,0.000231,0.000405
4,BCE1_D231_718303,2023-05-31,0.037333,0.037167,0.046562,0.0280,0.000926,0.000921,0.001154,0.000694,...,0.0,1,0.009333,0.020833,0.000231,0.000517,0.009333,0.016333,0.000231,0.000405
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
253311,MCW2_D463_811220,2026-11-30,0.000000,0.000000,0.000000,0.0000,0.000000,0.000000,0.000000,0.000000,...,0.0,0,0.023667,0.023333,0.000835,0.000824,NaN,NaN,NaN,NaN
253312,MCW2_D463_811220,2026-12-31,0.000000,0.000000,0.000000,0.0000,0.000000,0.000000,0.000000,0.000000,...,0.0,0,0.023667,0.023333,0.000835,0.000824,NaN,NaN,NaN,NaN
253313,MCW2_D463_811220,2027-01-31,0.000000,0.000000,0.000000,0.0000,0.000000,0.000000,0.000000,0.000000,...,0.0,0,0.023667,0.023333,0.000835,0.000824,NaN,NaN,NaN,NaN
253314,MCW2_D463_811220,2027-02-28,0.000000,0.000000,0.000000,0.0000,0.000000,0.000000,0.000000,0.000000,...,0.0,0,0.023667,0.023333,0.000835,0.000824,NaN,NaN,NaN,NaN


In [30]:
# all_df[(all_df['month']==4) & (all_df['final_seasonal_month'] == 1)]['brand_code'].unique()

In [31]:
all_df[(all_df['final_seasonal_month'] == 1)]['brand_code'].unique()

array(['REV.LQDST', 'PCNO(R)', 'REV.ST.', 'PADV-HOT', 'SF_SOYACN',
       'SAF_HONEY', 'PA-BDYLOT', 'SW STLDEO', 'NHR_SSAHO', 'PA_CN_HO'],
      dtype=object)

### co p3m

In [32]:
# query = """select * from TRN_MIL_CO_DATA where month_date >= '2025-01-31'"""
# co_df = pd.read_sql(query, con=dev_conn)
# co_df

In [33]:
# co_df.columns = co_df.columns.str.lower()
# co_df.columns

In [34]:
# all_df = pd.read_csv('break.csv')
# all_df

In [35]:
# all_df['co_flag'] = ((all_df['extra_vol_co'] == 1) | (all_df['other_co'] == 1) | (all_df['price_off_co'] == 1)).astype(int)
# all_df

In [36]:
# all_df[all_df['co_flag'] == 1]

In [37]:
# co_df = co_df[co_df['co_flag'] == 1]
# co_df = co_df[co_df['channel'] == 'GT']
# co_df

In [38]:
# co_df['asm_area_code'].unique()

In [39]:
# realignment_df = pd.read_sql(
#     'select * from trn_mil_asm_psku_realignment',
#     dev_conn
# )
# realignment_df.columns = realignment_df.columns.str.lower()

# def demand_driver_realign_pskus(data):
#         """
#         Realign the old pskus to new pskus and return updated data.

#         Args:
#             data: pandas dataframe
#             - master dataframe having all_df the pskus
        
#         Return:
#             data: pandas dataframe
#             - dataframe 
#         """
#         realignment_data = realignment_df.copy()
#         realignment_data.columns = realignment_data.columns.str.lower()
#         realignment_data = realignment_data[
#             (realignment_data["channel"] == 'GT')
#             | (realignment_data["channel"] == 'GT' + " B2C")
#             | (realignment_data["channel"] == "ALL")
#         ]

#         data["parent_material_code"] = data["parent_material_code"].astype(int)

#         for grp, grp_data in realignment_data.groupby(by=["psku old", "asm"]):
#             old_psku, old_asm = grp
#             new_psku = grp_data["psku new"].values[0]
#             if old_asm != "ALL":
#                 condition = (data["parent_material_code"] == old_psku) & (
#                     data["asm_area_code"] == old_asm
#                 )
#             else:
#                 condition = data["parent_material_code"] == old_psku

#             data.loc[condition, "parent_material_code"] = new_psku

#         return data

In [40]:
# co_data = co_df.rename( columns= {'regionhierarchy':'asm_area_code'})
# co_data = co_data.replace( { True:1 , False:0})
# co_data = co_data[[ "month_date", 'asm_area_code', "parent_material_code", 'extra_vol_co', 'price_off_co', 'other_co']]
# co_data = demand_driver_realign_pskus(co_data)
# co_data = co_data.groupby(["month_date", 'asm_area_code', "parent_material_code"], as_index=False).max()
# co_data["month_date"] = pd.to_datetime(co_data["month_date"])


# ### (old_asm, new_asm)
# # asm_change_info = [('HUB', 'KARC'), ('HUB', 'KARN'), ('WUP', 'NUP'), ('TPT', 'KUR')]
# asm_change_info = {
#     'KARC': {'old_asm': 'HUB', 'data_available_from': '2025-01-31'},
#     'KARN': {'old_asm': 'HUB', 'data_available_from': '2025-01-31'},
#     'NUP': {'old_asm': 'WUP', 'data_available_from': '2025-01-31'},
#     'KUR': {'old_asm': 'TPT', 'data_available_from': None}
# }


# for new_asm in asm_change_info.keys():
#     old_asm = asm_change_info[new_asm]['old_asm']
#     data_available_from = asm_change_info[new_asm]['data_available_from']

#     if data_available_from is not None:
#         old_asm_df = co_data[
#             (co_data['asm_area_code'] == old_asm) &
#             (co_data['month_date'] < pd.to_datetime(data_available_from))
#         ]
#     else:
#         old_asm_df = co_data[
#             (co_data['asm_area_code'] == old_asm)
#         ]

#     new_asm_df = old_asm_df.copy()
#     new_asm_df['asm_area_code'] = new_asm
#     co_data = pd.concat([co_data, new_asm_df])
    
# co_data = co_data.drop_duplicates()
# assert co_data.duplicated(
#     subset=['asm_area_code', 'parent_material_code', 'month_date']
# ).sum() == 0
# ###

# # co_data = co_data.reset_index(drop=True)
# co_data = co_data.fillna(0)
# all_df["month_date"] = pd.to_datetime(all_df["month_date"])

# all_df = pd.merge(
#     all_df,
#     co_data,
#     on=["month_date", 'asm_area_code',"parent_material_code"],
#     how="left",
# )

In [41]:
# all_df

In [42]:
# all_df = all_df.fillna(0)
# all_df['co_flag'] = ((all_df['extra_vol_co_y'] == 1) | (all_df['other_co_y'] == 1) | (all_df['price_off_co_y'] == 1)).astype(int)
# all_df[all_df['co_flag'] == 1]

In [43]:
all_df.columns[:60]

Index(['key', 'month_date', 'pred_p3m', 'pred_p6m', 'pred_prophet', 'pred_rf',
       'pred_value_p3m', 'pred_value_p6m', 'pred_value_prophet',
       'pred_value_rf', 'channel', 'asm_area_code', 'depot_code',
       'parent_material_code', 'brand_code', 'diwali', 'diwali_lead_1',
       'diwali_lead_2', 'ganesh_chaturthi', 'ganesh_chaturthi_lead_1',
       'ganesh_chaturthi_lead_2', 'ratio_last_year', 'quarter',
       'sec_vol_actuals_rum_month_value', 'pred_best_model',
       'pred_value_best_model', 'sec_vol_actuals_rum_month_treated',
       'sec_vol_actuals_rum_month_value_treated', 'train_till', 'cov', 'run',
       'step', 'file_path', 'run_month', 'M month', 'pred_prophet_60%ile',
       'pred_prophet_70%ile', 'portfolio', 'qtr_ind_rate',
       'sec_vol_actuals_rum_month', 'P3M', 'P6M', 'LY P3M', 'LY P6M',
       'LY P3M_copy', 'P3M Max', 'P3M Top 2 Mean', 'MoM P3M growth',
       'MoM P3M growth_lag_1', 'MoM P3M growth_lag_2',
       '>=20%_3M_inc_month_count', 'Avg(P3M Mea

In [44]:
# all_df.rename(columns = {'run_month_x':'run_month'}, inplace = True)

In [45]:
# all_df['run_month'] = pd.to_datetime(all_df['run_month'])
# def compute_adjusted_pm(df_grp, window, column, year_shift=0):
#     df_grp = df_grp.sort_values("month_date")

#     run_month = df_grp["run_month"].iloc[0]

#     # define cutoff
#     end_date = run_month - pd.DateOffset(years=year_shift)

#     # keep only eligible history (before run month & non-event)
#     hist = df_grp[
#         (df_grp["month_date"] < end_date) &
#         (df_grp["co_flag"] == 0)
#     ]

#     if hist.empty:
#         return np.nan

#     # take last `window` non-event months
#     hist = hist.tail(window)

#     # if len(hist) < window:
#     #     return np.nan   # optional, keeps behavior strict

#     return hist[column].mean()

# adj_df = all_df.groupby(
#     ['key', "run_month"]
# ).apply(
#     lambda x: pd.Series({
#         "P3M_non_co": compute_adjusted_pm(x, 3,'sec_vol_actuals_rum_month',0),
#         "P6M_non_co": compute_adjusted_pm(x, 6,'sec_vol_actuals_rum_month',0),
#         "P3M_non_co_value": compute_adjusted_pm(x, 3,'sec_vol_actuals_rum_month_value',0),
#         "P6M_non_co_value": compute_adjusted_pm(x, 6,'sec_vol_actuals_rum_month_value',0)
#     })
# ).reset_index()
# adj_df


# # adj_ly_df = all_df.groupby(
# #     ['key', "run_month"]
# # ).apply(
# #     lambda x: pd.Series({
# #         "LY_P3M_non_seasonal": compute_adjusted_pm(x, 3,'sec_vol_actuals_rum_month',year_shift=1),
# #         "LY_P6M_non_seasonal": compute_adjusted_pm(x, 6,'sec_vol_actuals_rum_month', year_shift=1),
# #         "LY_P3M_non_seasonal_value": compute_adjusted_pm(x, 3,'sec_vol_actuals_rum_month_value', year_shift=1),
# #         "LY_P6M_non_seasonal_value": compute_adjusted_pm(x, 6,"sec_vol_actuals_rum_month_value", year_shift=1)
# #     })
# # ).reset_index()

# # adj_df = adj_df.merge(adj_ly_df, on = ['key', 'run_month'], how = 'left')
# #adj_df[adj_df['key'] == 'reliance_b2c_2_haryana_718488']
# adj_df

In [46]:
# all_df = all_df.merge(adj_df, on=['key', 'run_month'], how = 'left')

In [47]:
# all_df.columns[:60]

In [48]:
# adj_df.to_csv('non_co_p3m_apr.csv')

### co p3m end

In [49]:
all_df['M month'].unique()
all_df[all_df['M month']!='0']

,key,month_date,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,...,is_seasonal_month_psku,final_seasonal_month,P3M_non_seasonal,P6M_non_seasonal,P3M_non_seasonal_value,P6M_non_seasonal_value,LY_P3M_non_seasonal,LY_P6M_non_seasonal,LY_P3M_non_seasonal_value,LY_P6M_non_seasonal_value
43,BCE1_D231_718303,2026-08-31,0.023333,0.018667,0.020496,0.0209,0.000578,0.000463,0.000508,0.000518,...,0.0,0,0.009333,0.020833,0.000231,0.000517,0.009333,0.016333,0.000231,0.000405
44,BCE1_D231_718303,2026-09-30,0.023333,0.018667,0.000000,0.0210,0.000578,0.000463,0.000000,0.000521,...,0.0,0,0.009333,0.020833,0.000231,0.000517,0.009333,0.016333,0.000231,0.000405
45,BCE1_D231_718303,2026-10-31,0.023333,0.018667,0.010294,0.0180,0.000578,0.000463,0.000255,0.000446,...,0.0,0,0.009333,0.020833,0.000231,0.000517,0.009333,0.016333,0.000231,0.000405
46,BCE1_D231_718303,2026-11-30,0.023333,0.018667,0.008829,0.0346,0.000578,0.000463,0.000219,0.000858,...,0.0,0,0.009333,0.020833,0.000231,0.000517,0.009333,0.016333,0.000231,0.000405
47,BCE1_D231_718303,2026-12-31,0.023333,0.018667,0.006594,0.0221,0.000578,0.000463,0.000163,0.000548,...,0.0,0,0.009333,0.020833,0.000231,0.000517,0.009333,0.016333,0.000231,0.000405
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
253311,MCW2_D463_811220,2026-11-30,0.000000,0.000000,0.000000,0.0000,0.000000,0.000000,0.000000,0.000000,...,0.0,0,0.023667,0.023333,0.000835,0.000824,NaN,NaN,NaN,NaN
253312,MCW2_D463_811220,2026-12-31,0.000000,0.000000,0.000000,0.0000,0.000000,0.000000,0.000000,0.000000,...,0.0,0,0.023667,0.023333,0.000835,0.000824,NaN,NaN,NaN,NaN
253313,MCW2_D463_811220,2027-01-31,0.000000,0.000000,0.000000,0.0000,0.000000,0.000000,0.000000,0.000000,...,0.0,0,0.023667,0.023333,0.000835,0.000824,NaN,NaN,NaN,NaN
253314,MCW2_D463_811220,2027-02-28,0.000000,0.000000,0.000000,0.0000,0.000000,0.000000,0.000000,0.000000,...,0.0,0,0.023667,0.023333,0.000835,0.000824,NaN,NaN,NaN,NaN


### Current month different logic

In [50]:
# all_df = pd.read_excel("/data/aman_singh/acuuracy_check/all_combination_GT_channel_trend2.xlsx", sheet_name='all_combination_GT_channel_tren')
# all_df

In [51]:
x = all_df.copy()

In [52]:
all_brand = all_df.groupby(['channel','brand_code', 'run_month','month_date'])['sec_vol_actuals_rum_month_value'].sum().reset_index()
all_brand

,channel,brand_code,run_month,month_date,sec_vol_actuals_rum_month_value
0,MT,4700_BCPC,2026-08-31,2026-07-31,0.0
1,MT,4700_BCPC,2026-08-31,2026-08-31,0.0
2,MT,4700_BCPC,2026-08-31,2026-09-30,0.0
3,MT,4700_BCPC,2026-08-31,2026-10-31,0.0
4,MT,4700_BCPC,2026-08-31,2026-11-30,0.0
...,...,...,...,...,...
4107,MT,TRU_ELMNT,2026-08-31,2026-11-30,0.0
4108,MT,TRU_ELMNT,2026-08-31,2026-12-31,0.0
4109,MT,TRU_ELMNT,2026-08-31,2027-01-31,0.0
4110,MT,TRU_ELMNT,2026-08-31,2027-02-28,0.0


In [53]:
all_brand['sec_vol_actuals_rum_month_value'].sum()

5083.444897154152

In [ ]:

# def detect_month_anomaly(df, brand_code, month_num, mon=9,threshold=0.25, months_window=3):
#     """
#     Detect if a specific month's sec_vol_actuals_rum_month_value is >25% different 
#     from past 3 months & next 3 months, and if pattern repeats in last 2 years.
    
#     Parameters:
#     - df: input dataframe with 'month_date', 'sec_vol_actuals_rum_month_value'
#     - brand_code: filter by this brand code
#     - month_num: month to check (6, 7, 8, 9)
#     - threshold: 25% difference threshold
#     - months_window: number of months before and after to compare
#     """
    
#     df_brand = df[df['brand_code'] == brand_code].sort_values('month_date').copy()
    
#     if df_brand.empty:
#         return None
    
#     df_brand['year'] = df_brand['month_date'].dt.year
#     df_brand['month'] = df_brand['month_date'].dt.month
    
#     years = sorted(df_brand['year'].unique())
#     current_year = years[-1]
#     past_years = [y for y in years if y < current_year][-2:]
    
#     anomalies = []
    
#     for year in past_years:
#         df_year = df_brand[df_brand['year'] == year].sort_values('month_date')
        
#         month_data = df_year[df_year['month'] == month_num]
#         if month_data.empty:
#             continue
        
#         month_value = month_data['sec_vol_actuals_rum_month_value'].iloc[0]
        
#         past_months = [(mon - i - 1) % 12 for i in range(1, months_window + 1)]
#         print(past_months)
#         next_months = [(mon + i - 1) % 12 + 1 for i in range(1, months_window + 1)]
        
#         past_m = df_year[df_year['month'].isin(past_months)]['sec_vol_actuals_rum_month_value']
#         next_m = df_year[df_year['month'].isin(next_months)]['sec_vol_actuals_rum_month_value']
        
#         comparison_values = pd.concat([past_m])
        
#         if comparison_values.empty:
#             continue
        
#         pct_diffs = []
#         for comp_value in comparison_values:
#             if comp_value != 0:
#                 pct_diff = (month_value - comp_value) / comp_value
#                 pct_diffs.append(pct_diff)
        
#         if pct_diffs:
#             positive_diffs = [p for p in pct_diffs if p > 0]
#             negative_diffs = [p for p in pct_diffs if p < 0]
#             same_sign = len(positive_diffs) == len(pct_diffs) or len(negative_diffs) == len(pct_diffs)
#             is_anomaly = len([p for p in pct_diffs if abs(p) > threshold]) == len(pct_diffs) and same_sign
#         else:
#             is_anomaly = False

#         anomalies.append({
#             'brand_code': brand_code,
#             'month': month_num,
#             'year': year,
#             'month_value': month_value,
#             'num_months_compared': len(comparison_values),
#             'pct_diffs_from_each': pct_diffs,
#             'min_pct_diff': min(pct_diffs) * 100 if pct_diffs else None,
#             'max_pct_diff': max(pct_diffs) * 100 if pct_diffs else None,
#             'is_anomaly': is_anomaly,
#             'direction': 'higher' if month_value > comparison_values.mean() else 'lower'
#         })
    
#     if len(anomalies) == 2:
#         pattern_repeats = anomalies[0]['is_anomaly'] and anomalies[1]['is_anomaly']
#         return pd.DataFrame(anomalies), pattern_repeats
    
#     return pd.DataFrame(anomalies), False


# # Check months 6, 7, 8, 9
# brands = all_brand['brand_code'].unique()
# results = []

# for month in [9,10,11,12]:
#     for brand in brands:
#         df_result, repeats = detect_month_anomaly(all_brand, brand, month)
#         if df_result is not None and not df_result.empty:
#             df_result['pattern_repeats'] = repeats
#             results.append(df_result)

# anomaly_summary = pd.concat(results, ignore_index=True)
# # print(anomaly_summary[anomaly_summary['pattern_repeats'] == True])

In [ ]:
all_brand

In [54]:
import pandas as pd


def detect_month_anomaly(
    df,
    brand_code,
    threshold=0.25,
    months_window=3,
    future_months=4
):
    """
    Logic:

    1. Get run_month dynamically.
    2. Analyze the next 4 months after run_month.
    3. For each future month:
         - Look at that month's value in Last Year (LY)
         - Compare it with the 3 months immediately preceding
           the run_month in LY.
    4. Do the exact same comparison for 2 Years Ago.
    5. A future month is finalized as an anomaly only when
       BOTH LY and 2Y-ago show the anomaly.
    6. For a given year, all_df 3 percentage differences must:
         - exceed 25%
         - have the same direction
    """

    # =========================================================
    # 1. Filter brand
    # =========================================================

    df_brand = df[
        df['brand_code'] == brand_code
    ].copy()

    if df_brand.empty:
        return None, False

    # =========================================================
    # 2. Convert dates
    # =========================================================

    df_brand['month_date'] = pd.to_datetime(
        df_brand['month_date']
    )

    df_brand['run_month'] = pd.to_datetime(
        df_brand['run_month']
    )

    # Normalize month_date to month-end so that
    # comparisons work consistently.
    df_brand['month_date'] = (
        df_brand['month_date']
        + pd.offsets.MonthEnd(0)
    )

    # =========================================================
    # 3. Determine run month dynamically
    # =========================================================

    run_month = df_brand['run_month'].max()

    run_month = (
        pd.Timestamp(run_month)
        + pd.offsets.MonthEnd(0)
    )

    # =========================================================
    # 4. Determine next 4 months
    # =========================================================

    target_dates = [
        run_month + pd.DateOffset(months=i)
        for i in range(1, future_months + 1)
    ]

    # Example:
    #
    # run_month = Sep 2026
    #
    # target_dates:
    # Oct 2026
    # Nov 2026
    # Dec 2026
    # Jan 2027

    # =========================================================
    # 5. Historical years
    # =========================================================

    # For each target month we evaluate:
    #
    # LY
    # 2Y ago
    #
    # Example:
    #
    # Oct 2026
    #   -> Oct 2025
    #   -> Oct 2024

    historical_offsets = [1, 2]

    results = []

    # =========================================================
    # 6. Process each future month
    # =========================================================

    for target_date in target_dates:

        year_results = {}

        # -----------------------------------------------------
        # For LY and 2Y ago
        # -----------------------------------------------------

        for year_offset in historical_offsets:

            # -------------------------------------------------
            # Target historical date
            # -------------------------------------------------

            historical_target_date = (
                target_date
                - pd.DateOffset(years=year_offset)
            )

            # -------------------------------------------------
            # Fixed comparison months
            #
            # These are based on the run month.
            #
            # Example:
            #
            # run_month = Sep 2026
            #
            # LY:
            # target = Oct 2025
            # comparison = Jul, Jun, May 2025
            #
            # 2Y:
            # target = Oct 2024
            # comparison = Jul, Jun, May 2024
            # -------------------------------------------------

            comparison_dates = [
                (
                    run_month
                    - pd.DateOffset(months=i)
                    - pd.DateOffset(years=year_offset)
                )
                for i in range(
                    1,
                    months_window + 1
                )
            ]

            # -------------------------------------------------
            # Get target value
            # -------------------------------------------------

            target_value_series = df_brand.loc[
                df_brand['month_date'] == historical_target_date,
                'sec_vol_actuals_rum_month_value'
            ]

            if target_value_series.empty:
                year_results[year_offset] = None
                continue

            target_value = target_value_series.iloc[0]

            # -------------------------------------------------
            # Get comparison values
            # -------------------------------------------------

            comparison_values = []

            missing_comparison = False

            for comparison_date in comparison_dates:

                value_series = df_brand.loc[
                    df_brand['month_date'] == comparison_date,
                    'sec_vol_actuals_rum_month_value'
                ]

                if value_series.empty:
                    missing_comparison = True
                    break

                comparison_values.append(
                    value_series.iloc[0]
                )

            if missing_comparison:
                year_results[year_offset] = None
                continue

            # -------------------------------------------------
            # Calculate percentage differences
            # -------------------------------------------------

            pct_diffs = []

            for comparison_value in comparison_values:

                if comparison_value == 0:
                    continue

                pct_diff = (
                    (target_value - comparison_value)
                    / comparison_value
                )

                pct_diffs.append(pct_diff)

            # Need all_df 3 comparisons
            if len(pct_diffs) != months_window:

                year_results[year_offset] = None
                continue

            # -------------------------------------------------
            # Check same direction
            # -------------------------------------------------

            all_positive = all(
                p > 0 for p in pct_diffs
            )

            all_negative = all(
                p < 0 for p in pct_diffs
            )

            same_direction = (
                all_positive or all_negative
            )

            # -------------------------------------------------
            # Check threshold
            # -------------------------------------------------

            all_above_threshold = all(
                abs(p) > threshold
                for p in pct_diffs
            )

            # -------------------------------------------------
            # Final anomaly for this historical year
            # -------------------------------------------------

            is_anomaly = (
                same_direction
                and all_above_threshold
            )

            direction = None

            if all_positive:
                direction = 'higher'

            elif all_negative:
                direction = 'lower'

            # -------------------------------------------------
            # Store year result
            # -------------------------------------------------

            year_results[year_offset] = {
                'historical_target_date': historical_target_date,
                'target_value': target_value,

                'comparison_dates': comparison_dates,
                'comparison_values': comparison_values,

                'pct_diffs': pct_diffs,

                'min_pct_diff': min(pct_diffs) * 100,
                'max_pct_diff': max(pct_diffs) * 100,

                'same_direction': same_direction,
                'all_above_threshold': all_above_threshold,

                'is_anomaly': is_anomaly,
                'direction': direction
            }

        # =====================================================
        # 7. Final anomaly requires BOTH years
        # =====================================================

        ly_result = year_results.get(1)
        two_year_result = year_results.get(2)

        pattern_repeats = (
            ly_result is not None
            and two_year_result is not None
            and ly_result['is_anomaly']
            and two_year_result['is_anomaly']
        )

        # =====================================================
        # 8. Store final result
        # =====================================================

        row = {
            'brand_code': brand_code,

            'run_month': run_month,

            'target_month': target_date,

            'target_year': target_date.year,

            # -----------------------------
            # Last Year
            # -----------------------------

            'ly_target_date': (
                ly_result['historical_target_date']
                if ly_result else None
            ),

            'ly_target_value': (
                ly_result['target_value']
                if ly_result else None
            ),

            'ly_comparison_dates': (
                ly_result['comparison_dates']
                if ly_result else None
            ),

            'ly_comparison_values': (
                ly_result['comparison_values']
                if ly_result else None
            ),

            'ly_pct_diffs': (
                ly_result['pct_diffs']
                if ly_result else None
            ),

            'ly_min_pct_diff': (
                ly_result['min_pct_diff']
                if ly_result else None
            ),

            'ly_max_pct_diff': (
                ly_result['max_pct_diff']
                if ly_result else None
            ),

            'ly_direction': (
                ly_result['direction']
                if ly_result else None
            ),

            'ly_is_anomaly': (
                ly_result['is_anomaly']
                if ly_result else False
            ),

            # -----------------------------
            # 2 Years Ago
            # -----------------------------

            'two_year_target_date': (
                two_year_result['historical_target_date']
                if two_year_result else None
            ),

            'two_year_target_value': (
                two_year_result['target_value']
                if two_year_result else None
            ),

            'two_year_comparison_dates': (
                two_year_result['comparison_dates']
                if two_year_result else None
            ),

            'two_year_comparison_values': (
                two_year_result['comparison_values']
                if two_year_result else None
            ),

            'two_year_pct_diffs': (
                two_year_result['pct_diffs']
                if two_year_result else None
            ),

            'two_year_min_pct_diff': (
                two_year_result['min_pct_diff']
                if two_year_result else None
            ),

            'two_year_max_pct_diff': (
                two_year_result['max_pct_diff']
                if two_year_result else None
            ),

            'two_year_direction': (
                two_year_result['direction']
                if two_year_result else None
            ),

            'two_year_is_anomaly': (
                two_year_result['is_anomaly']
                if two_year_result else False
            ),

            # -----------------------------
            # FINAL
            # -----------------------------

            'pattern_repeats': pattern_repeats,

            'is_final_anomaly': pattern_repeats
        }

        results.append(row)

    # =========================================================
    # 9. Return
    # =========================================================

    result_df = pd.DataFrame(results)

    return result_df, result_df['is_final_anomaly'].any()


# =============================================================
# RUN FOR ALL BRANDS
# =============================================================

brands = all_brand['brand_code'].unique()

results = []

for brand in brands:

    df_result, repeats = detect_month_anomaly(
        all_brand,
        brand_code=brand,
        threshold=0.25,
        months_window=3,
        future_months=4
    )

    if df_result is not None and not df_result.empty:
        results.append(df_result)


# =============================================================
# FINAL OUTPUT
# =============================================================

if results:

    anomaly_summary = pd.concat(
        results,
        ignore_index=True
    )

else:

    anomaly_summary = pd.DataFrame()

/tmp/ipykernel_3195255/1207719152.py:464: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  anomaly_summary = pd.concat(


In [57]:
anomaly_summary.to_csv('brand_month_anomaly_summary.csv', index=False)

In [ ]:
anomaly_summary.rename(columns={'target_month':'month'}, inplace=True)


In [1]:
all

<function all(iterable, /)>

In [59]:
all_df.shape

(253316, 130)

In [65]:
final_brands = anomaly_summary[anomaly_summary['pattern_repeats'] == True].drop_duplicates(subset=['brand_code','month_date'])[['brand_code', 'month_date','two_year_direction']]
final_brands['month_different'] = 1
all_df['month'] = all_df['month_date'].dt.month
all_df = all_df.merge(final_brands[['brand_code', 'month_date','month_different']], on = ['brand_code','month_date'], how = 'left')
all_df['month_different'].fillna(0, inplace=True)
all_df

,key,month_date,pred_p3m,pred_p6m,pred_prophet,pred_rf,pred_value_p3m,pred_value_p6m,pred_value_prophet,pred_value_rf,...,final_seasonal_month,P3M_non_seasonal,P6M_non_seasonal,P3M_non_seasonal_value,P6M_non_seasonal_value,LY_P3M_non_seasonal,LY_P6M_non_seasonal,LY_P3M_non_seasonal_value,LY_P6M_non_seasonal_value,month_different
0,BCE1_D231_718303,2023-01-31,0.018667,0.037167,0.041426,0.0112,0.000463,0.000921,0.001027,0.000278,...,0,0.009333,0.020833,0.000231,0.000517,0.009333,0.016333,0.000231,0.000405,0.0
1,BCE1_D231_718303,2023-02-28,0.018667,0.037167,0.039172,0.0084,0.000463,0.000921,0.000971,0.000208,...,0,0.009333,0.020833,0.000231,0.000517,0.009333,0.016333,0.000231,0.000405,0.0
2,BCE1_D231_718303,2023-03-31,0.018667,0.037167,0.039450,0.0140,0.000463,0.000921,0.000978,0.000347,...,1,0.009333,0.020833,0.000231,0.000517,0.009333,0.016333,0.000231,0.000405,0.0
3,BCE1_D231_718303,2023-04-30,0.018667,0.037167,0.042242,0.0182,0.000463,0.000921,0.001047,0.000451,...,1,0.009333,0.020833,0.000231,0.000517,0.009333,0.016333,0.000231,0.000405,0.0
4,BCE1_D231_718303,2023-05-31,0.037333,0.037167,0.046562,0.0280,0.000926,0.000921,0.001154,0.000694,...,1,0.009333,0.020833,0.000231,0.000517,0.009333,0.016333,0.000231,0.000405,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
253311,MCW2_D463_811220,2026-11-30,0.000000,0.000000,0.000000,0.0000,0.000000,0.000000,0.000000,0.000000,...,0,0.023667,0.023333,0.000835,0.000824,NaN,NaN,NaN,NaN,0.0
253312,MCW2_D463_811220,2026-12-31,0.000000,0.000000,0.000000,0.0000,0.000000,0.000000,0.000000,0.000000,...,0,0.023667,0.023333,0.000835,0.000824,NaN,NaN,NaN,NaN,0.0
253313,MCW2_D463_811220,2027-01-31,0.000000,0.000000,0.000000,0.0000,0.000000,0.000000,0.000000,0.000000,...,0,0.023667,0.023333,0.000835,0.000824,NaN,NaN,NaN,NaN,0.0
253314,MCW2_D463_811220,2027-02-28,0.000000,0.000000,0.000000,0.0000,0.000000,0.000000,0.000000,0.000000,...,0,0.023667,0.023333,0.000835,0.000824,NaN,NaN,NaN,NaN,0.0


In [66]:
all_df[all_df['month_date']=='2026-08-31']['P3M_value'].sum()

129.85475414886434

In [ ]:
all_df[all_df['M month'] != '0'].to_csv('/data/aman_singh/acuuracy_check/all_combination_MT_live_aug.csv')

In [ ]:
all_df[all_df['month_date']<='2026-07-31'].to_csv('/data/aman_singh/acuuracy_check/all_combination_MT_channel_trend.csv')

In [ ]:
# all_df[all_df['month_date'] == '2026-08-31']['pred_value_rf'].sum()

In [ ]:
# with pd.ExcelWriter('all_combination_MT_live_apr.xlsx', engine='xlsxwriter') as writer:
#     all_df[all_df['M month'] != '0'].to_excel(
#         writer,
#         sheet_name='Base',
#         index=False
#     )
    
#     trend_df.to_excel(
#         writer,
#         sheet_name='threshold',
#         index=False
#     )
    
    


In [ ]:
# all_df[all_df['M month'].notna()].to_csv('/data/aman_singh/acuuracy_check/all_combination_GT_live_july.csv')

In [ ]:
# import pandas as pd
# all_df = pd.read_csv('/data/aman_singh/acuuracy_check/all_combination_all_channels_pred3.csv')
# all_df